# JSE Combined Decision System
## Fundamentals + Technicals + Risk → Actionable Investment Decisions

This notebook merges:
- **TradingView Snapshot** (fundamentals: EPS, ROE, margins, debt)
- **Yahoo Finance Historical** (technicals: RSI, MACD, momentum, Sharpe)
- **Risk Metrics** (drawdown, volatility, Sharpe ratio)

**Scoring Weights:** 50% Fundamentals | 30% Technicals | 20% Risk


In [1]:
import sys, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 50)

# ── CONFIGURATION ──
PROJECT_DIR = Path(r"C:\Users\chris\Desktop\South_African_Stocks")
SNAPSHOT_DATE = "2026-05-15"   # Update when you upload new data
MIN_LIQUIDITY = 1_000_000      # R1M/day minimum trading value
RISK_FREE_RATE = 0.08          # 8% SA risk-free rate

HIST_DIR = PROJECT_DIR / "data" / "historical"
SNAP_DIR = PROJECT_DIR / "data" / "snapshots" / SNAPSHOT_DATE
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Project: {PROJECT_DIR}")
print(f"Snapshot: {SNAPSHOT_DATE}")
print(f"Min Liquidity: R{MIN_LIQUIDITY:,.0f}/day")


Project: C:\Users\chris\Desktop\South_African_Stocks
Snapshot: 2026-05-15
Min Liquidity: R1,000,000/day


## 1. Load Fundamentals (TradingView Snapshot)


In [2]:
snap = pd.read_parquet(SNAP_DIR / "snapshot.parquet")
print(f"Loaded {len(snap)} stocks from TradingView snapshot")
snap[["symbol","sector","price","pe_ratio","eps_growth_ttm","roe_ttm","net_margin_ttm","debt_to_equity"]].head(10)


Loaded 245 stocks from TradingView snapshot


,symbol,sector,price,pe_ratio,eps_growth_ttm,roe_ttm,net_margin_ttm,debt_to_equity
0,BHG,Non-energy minerals,70265.0,20.258822,-13.276997,21.278042,18.951951,0.555895
1,ANH,Consumer non-durables,133707.0,22.656362,13.244074,8.119661,11.583629,0.836631
2,BTI,Consumer non-durables,108984.0,13.903356,144.645368,15.736556,29.976572,0.731753
3,CFR,Consumer durables,328291.0,22.985884,226.959292,18.202412,17.452673,0.667613
4,GLN,Distribution services,12732.0,268.740642,NaN,0.831351,0.135978,1.067576
5,PRX,Technology services,76368.0,7.092256,91.409327,27.250552,198.640715,0.321040
6,AGL,Non-energy minerals,84065.0,NaN,-16.886532,-5.880417,-6.261558,0.861252
7,ANG,Non-energy minerals,158759.0,13.671263,NaN,45.385068,31.112305,0.267838
8,NPN,Technology services,86489.0,6.884510,85.428004,26.660247,73.009276,0.748373
9,GFI,Non-energy minerals,68648.0,9.849286,177.841859,52.992345,40.510650,0.381994


## 2. Compute Technicals (5-Year Historical Data)


In [3]:
def compute_technicals(hist_dir):
    rows = []
    for f in sorted(hist_dir.glob("*.parquet")):
        sym = f.stem
        if sym.startswith(("download_", "bulk_")):
            continue
        try:
            df = pd.read_parquet(f)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            close = df["Close"].squeeze()
            if len(close) < 200:
                continue
            price = float(close.iloc[-1])
            ret = close.pct_change().dropna()
            sma50 = float(close.rolling(50).mean().iloc[-1])
            sma200 = float(close.rolling(200).mean().iloc[-1])
            # RSI
            delta = close.diff()
            gain = delta.where(delta > 0, 0).rolling(14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
            rsi = float((100 - 100 / (1 + gain / loss)).iloc[-1])
            # MACD
            macd_line = close.ewm(span=12).mean() - close.ewm(span=26).mean()
            sig_line = macd_line.ewm(span=9).mean()
            # Bollinger
            bb_mid = close.rolling(20).mean()
            bb_std = close.rolling(20).std()
            bb_up = float((bb_mid + 2*bb_std).iloc[-1])
            bb_lo = float((bb_mid - 2*bb_std).iloc[-1])
            bb_pos = (price - bb_lo) / (bb_up - bb_lo) if bb_up != bb_lo else 0.5
            # Momentum
            def mom(d):
                return (price / float(close.iloc[-d]) - 1) * 100 if len(close) >= d else np.nan
            # Risk
            n = min(252, len(ret))
            vol_1y = float(ret.iloc[-n:].std() * np.sqrt(252) * 100)
            rf = RISK_FREE_RATE / 252
            exc = ret.iloc[-n:] - rf
            sharpe = float(exc.mean()*252 / (ret.iloc[-n:].std()*np.sqrt(252))) if ret.iloc[-n:].std() > 0 else np.nan
            down = ret.iloc[-n:][ret.iloc[-n:] < 0]
            sortino = float(exc.mean()*252 / (down.std()*np.sqrt(252))) if len(down) > 0 and down.std() > 0 else np.nan
            pw = close.iloc[-n:]
            max_dd = float(((pw - pw.cummax()) / pw.cummax()).min() * 100)
            avg_vol = float(df["Volume"].iloc[-20:].mean()) if "Volume" in df.columns else 0
            rows.append({
                "symbol": sym, "hist_price": price, "sma_50": sma50, "sma_200": sma200,
                "above_sma50": price > sma50, "above_sma200": price > sma200,
                "t_rsi_14": rsi, "t_macd": float(macd_line.iloc[-1]),
                "t_macd_signal": float(sig_line.iloc[-1]),
                "t_macd_bullish": float(macd_line.iloc[-1]) > float(sig_line.iloc[-1]),
                "bb_position": bb_pos,
                "mom_1m": mom(21), "mom_3m": mom(63), "mom_6m": mom(126), "mom_12m": mom(252),
                "volatility_1y": vol_1y, "sharpe_1y": sharpe, "sortino_1y": sortino,
                "max_drawdown_1y": max_dd, "avg_daily_value": avg_vol * price,
            })
        except Exception:
            pass
    return pd.DataFrame(rows)

tech = compute_technicals(HIST_DIR)
print(f"Computed technicals for {len(tech)} stocks")
tech[["symbol","hist_price","t_rsi_14","mom_12m","sharpe_1y","max_drawdown_1y"]].head(10)


Computed technicals for 240 stocks


,symbol,hist_price,t_rsi_14,mom_12m,sharpe_1y,max_drawdown_1y
0,4SI,69.0,50.000000,-5.479452,-0.104622,-20.731707
1,ABG,23018.0,44.991463,35.494709,1.141602,-16.995454
2,ACL,129.0,43.750000,12.173913,0.367071,-36.781609
3,ACS,1085.0,72.000000,67.258329,1.008805,-36.842104
4,ACT,120.0,74.390244,-24.050633,0.138482,-60.000000
5,ADH,4357.0,53.701380,40.727988,1.200423,-8.620164
6,ADR,644.0,43.956044,57.250637,0.847186,-27.495150
7,AEG,440.0,54.166667,-26.174497,-0.631803,-43.378378
8,AEL,2150.0,45.744681,-6.477603,-0.370205,-29.818148
9,AFE,11404.0,67.013611,12.721717,0.225699,-22.038245


## 3. Merge Fundamentals + Technicals


In [4]:
merged = snap.merge(tech, on="symbol", how="inner")
merged["liquid"] = merged["avg_daily_value"] >= MIN_LIQUIDITY
print(f"Merged: {len(merged)} stocks | {merged['liquid'].sum()} liquid")


Merged: 240 stocks | 218 liquid


## 4. Combined Scoring System
| Component | Weight | Measures |
|---|---|---|
| **Fundamentals** | 50% | EPS growth, ROE, margins, debt, balance sheet |
| **Technicals** | 30% | Trend, momentum, RSI, MACD |
| **Risk** | 20% | Sharpe, drawdown, volatility |


In [5]:
def score_fundamentals(row):
    s, n = 0, 0
    eg = row.get("eps_growth_ttm")
    if pd.notna(eg): s += min(20, max(0, 10 + eg * 0.2)); n += 1
    roe = row.get("roe_ttm")
    if pd.notna(roe): s += min(20, max(0, roe * 0.8)); n += 1
    nm = row.get("net_margin_ttm")
    if pd.notna(nm): s += min(15, max(0, nm * 0.5)); n += 1
    rg = row.get("revenue_growth_ttm")
    if pd.notna(rg): s += min(15, max(0, 7.5 + rg * 0.3)); n += 1
    de = row.get("debt_to_equity")
    if pd.notna(de): s += max(0, 15 - de * 0.1); n += 1
    cr = row.get("current_ratio")
    if pd.notna(cr): s += min(15, cr * 5) if cr <= 3 else 10; n += 1
    return s if n >= 3 else np.nan

def score_technicals(row):
    s = 0
    if row.get("above_sma200") and row.get("above_sma50"): s += 25
    elif row.get("above_sma200"): s += 15
    elif row.get("above_sma50"): s += 10
    m = row.get("mom_12m")
    if pd.notna(m): s += min(25, max(0, 12.5 + m * 0.25))
    rsi = row.get("t_rsi_14")
    if pd.notna(rsi):
        if 40 <= rsi <= 60: s += 25
        elif 30 <= rsi < 40 or 60 < rsi <= 70: s += 18
        elif rsi < 30: s += 20
        else: s += 8
    if pd.notna(row.get("t_macd")) and pd.notna(row.get("t_macd_signal")):
        if row["t_macd"] > row["t_macd_signal"]: s += 25 if row["t_macd"] > 0 else 18
        else: s += 5
    return s

def score_risk(row):
    s = 0
    sh = row.get("sharpe_1y")
    if pd.notna(sh): s += min(35, max(0, 10 + sh * 12))
    dd = abs(row.get("max_drawdown_1y", -50))
    s += 35 if dd < 10 else 28 if dd < 20 else 20 if dd < 30 else 10 if dd < 45 else 3
    v = row.get("volatility_1y")
    if pd.notna(v): s += 30 if v < 20 else 25 if v < 30 else 18 if v < 40 else 10 if v < 55 else 3
    return s

merged["fundamental_score"] = merged.apply(score_fundamentals, axis=1)
merged["technical_score"] = merged.apply(score_technicals, axis=1)
merged["risk_score"] = merged.apply(score_risk, axis=1)
merged["combined_score"] = merged["fundamental_score"]*0.50 + merged["technical_score"]*0.30 + merged["risk_score"]*0.20

def get_signal(cs):
    if pd.isna(cs): return "NO DATA"
    if cs >= 65: return "STRONG BUY"
    if cs >= 55: return "BUY"
    if cs >= 45: return "HOLD"
    if cs >= 35: return "SELL"
    return "STRONG SELL"

merged["signal"] = merged["combined_score"].apply(get_signal)
valid = merged[merged["combined_score"].notna()].copy()
liquid = valid[valid["liquid"]].sort_values("combined_score", ascending=False)

print(f"Scored {len(valid)} stocks | {len(liquid)} liquid")
for sig in ["STRONG BUY","BUY","HOLD","SELL","STRONG SELL"]:
    print(f"  {sig:12s}: {(liquid['signal']==sig).sum():>3d}")


Scored 235 stocks | 215 liquid
  STRONG BUY  :  66
  BUY         :  59
  HOLD        :  50
  SELL        :  20
  STRONG SELL :  20


## 5. Actionable Buy List — Top 25


In [6]:
cols = ["symbol","description","sector","hist_price","fundamental_score","technical_score",
        "risk_score","combined_score","signal","mom_12m","sharpe_1y"]
top25 = liquid[cols].head(25).copy()
top25.columns = ["Symbol","Name","Sector","Price","Fund","Tech","Risk","Combined","Signal","Mom12m","Sharpe"]
top25.index = range(1, len(top25)+1)
top25.index.name = "Rank"
top25


,Symbol,Name,Sector,Price,Fund,Tech,Risk,Combined,Signal,Mom12m,Sharpe
Rank,,,,,,,,,,,
1,GND,Grindrod Limited,Transportation,2381.0,88.234853,83.000000,86.280535,86.273533,STRONG BUY,82.301380,1.940045
2,RES,Resilient REIT Limited,Finance,8369.0,75.129245,97.628396,85.991286,84.051398,STRONG BUY,40.513583,1.499274
3,EQU,Equites Property Fund Ltd ZAR,Finance,1814.0,80.268457,92.222409,74.049904,82.610932,STRONG BUY,18.889637,0.504159
4,YYLBEE,YeboYethu Limited,Finance,5400.0,89.918823,83.000000,61.453578,82.150127,STRONG BUY,116.117092,1.787798
5,SEA,Spear REIT Ltd.,Finance,1300.0,88.789492,76.030599,73.767965,81.957519,STRONG BUY,34.122397,0.897330
6,APH,Alphamin Resources Corp.,Non-energy minerals,1700.0,94.064766,80.000000,50.425597,81.117503,STRONG BUY,50.587470,0.868800
7,NPH,Northam Platinum Holdings Limited,Non-energy minerals,33474.0,93.331184,83.000000,44.066961,80.378984,STRONG BUY,149.818478,1.755580
8,PAN,Pan African Resources PLC,Non-energy minerals,3087.0,89.421207,70.000000,65.000000,78.710603,STRONG BUY,181.462241,2.292087
9,CFR,Compagnie Financiere Richemont SA,Consumer durables,330500.0,83.483752,87.281947,52.177498,78.361959,STRONG BUY,-0.872214,-0.235209


## 6. Avoid List — Bottom 10


In [7]:
bottom = liquid[cols].tail(10).copy()
bottom.columns = ["Symbol","Name","Sector","Price","Fund","Tech","Risk","Combined","Signal","Mom12m","Sharpe"]
bottom.index = range(1, len(bottom)+1)
bottom


,Symbol,Name,Sector,Price,Fund,Tech,Risk,Combined,Signal,Mom12m,Sharpe
1,AFT,Afrimat Limited,Non-energy minerals,3180.0,33.770713,33.237608,20.000000,30.856639,STRONG SELL,-37.049570,-1.103690
2,BLU,Blu Label Unlimited Group Ltd,Distribution services,842.0,29.571970,36.242740,22.301784,30.119164,STRONG SELL,-5.029039,-0.058185
3,ACT,AfroCentric Investment Corporation Limited,Retail trade,120.0,23.207998,49.487342,17.661785,29.982559,STRONG SELL,-24.050633,0.138482
4,SPG,Super Group Limited,Transportation,1633.0,28.680695,43.959819,9.276070,29.383507,STRONG SELL,-46.160726,-0.560328
5,MTA,Metair Investments Limited,Producer manufacturing,496.0,38.821809,27.261905,8.845092,29.358494,STRONG SELL,-40.952381,-0.596242
6,SPP,Spar Group Limited,Distribution services,6180.0,31.548923,30.704000,21.000000,29.185661,STRONG SELL,-47.184001,-2.206198
7,AEG,Aveng Limited,Industrial services,440.0,21.005524,35.956376,22.418360,25.773347,STRONG SELL,-26.174497,-0.631803
8,TMT,Trematon Capital Investments Limited,Finance,107.0,23.333831,33.235294,17.499427,25.137389,STRONG SELL,-37.058824,-0.458381
9,SAP,Sappi Limited,Process industries,1486.0,26.599969,30.000000,13.000000,24.899984,STRONG SELL,-55.388772,-1.519453
10,CPR,Copper 360 Limited,Non-energy minerals,49.0,31.006768,23.000000,6.000000,23.603384,STRONG SELL,-74.345550,-1.412012


## 7. Blue-Chip Quality (Combined Score)
Stocks passing ALL gates: Combined ≥ 60, ROE > 15%, EPS Growth > 0, D/E < 100, Sharpe > 0.5, MaxDD > -30%


In [8]:
bluechip = liquid[
    (liquid["combined_score"] >= 60) & (liquid["roe_ttm"] > 15) &
    (liquid["eps_growth_ttm"] > 0) & (liquid["debt_to_equity"] < 100) &
    (liquid["sharpe_1y"] > 0.5) & (liquid["max_drawdown_1y"] > -30)
].copy()
bc_cols = ["symbol","description","sector","hist_price","combined_score","signal",
           "eps_growth_ttm","roe_ttm","net_margin_ttm","debt_to_equity","sharpe_1y","max_drawdown_1y","mom_12m"]
bc_avail = [c for c in bc_cols if c in bluechip.columns]
print(f"Blue-Chip Quality: {len(bluechip)} stocks passed all gates")
bluechip[bc_avail].head(20)


Blue-Chip Quality: 33 stocks passed all gates


,symbol,description,sector,hist_price,combined_score,signal,eps_growth_ttm,roe_ttm,net_margin_ttm,debt_to_equity,sharpe_1y,max_drawdown_1y,mom_12m
87,GND,Grindrod Limited,Transportation,2381.0,86.273533,STRONG BUY,558.673469,21.859492,38.484585,0.334013,1.940045,-19.637675,82.301380
70,RES,Resilient REIT Limited,Finance,8369.0,84.051398,STRONG BUY,58.547813,18.627134,116.403758,0.522774,1.499274,-11.876968,40.513583
157,YYLBEE,YeboYethu Limited,Finance,5400.0,82.150127,STRONG BUY,693.630698,60.021475,301.984366,1.987204,1.787798,-25.454545,116.117092
131,SEA,Spear REIT Ltd.,Finance,1300.0,81.957519,STRONG BUY,45.801741,16.848058,85.976805,0.335339,0.897330,-13.037037,34.122397
76,APH,Alphamin Resources Corp.,Non-energy minerals,1700.0,81.117503,STRONG BUY,73.170946,45.711225,25.499841,0.096131,0.868800,-22.888889,50.587470
43,PAN,Pan African Resources PLC,Non-energy minerals,3087.0,78.710603,STRONG BUY,157.481751,43.482176,29.117408,0.195158,2.292087,-26.305131,181.462241
2,BTI,British American Tobacco p.l.c.,Consumer non-durables,108000.0,76.920879,STRONG BUY,144.645368,15.736556,29.976572,0.731753,1.408377,-16.359607,47.812231
169,ART,Argent Industrial Limited,Producer manufacturing,3760.0,76.800036,STRONG BUY,17.417230,15.278596,10.245587,0.074529,1.367681,-8.160281,48.741618
64,VKE,Vukile Property Fund Limited,Finance,2349.0,75.461280,STRONG BUY,134.048708,15.544506,78.741589,0.835103,0.652413,-16.084453,19.314293
9,GFI,Gold Fields Limited,Non-energy minerals,67824.0,74.433846,STRONG BUY,177.841859,52.992345,40.510650,0.381994,1.284883,-28.800312,82.977582


## 8. Hidden Gems (High Potential + Under-the-Radar)


In [9]:
top20_syms = liquid.nlargest(20, "avg_daily_value")["symbol"].tolist()
gems = liquid[
    (liquid["combined_score"] >= 55) & (~liquid["symbol"].isin(top20_syms)) &
    (liquid["mom_12m"] > 20) & (liquid["sharpe_1y"] > 0.5)
].copy()
gem_cols = ["symbol","description","sector","hist_price","combined_score","signal","mom_12m","sharpe_1y","max_drawdown_1y"]
gem_avail = [c for c in gem_cols if c in gems.columns]
print(f"Hidden Gems: {len(gems)} stocks")
gems[gem_avail].head(20)


Hidden Gems: 51 stocks


,symbol,description,sector,hist_price,combined_score,signal,mom_12m,sharpe_1y,max_drawdown_1y
87,GND,Grindrod Limited,Transportation,2381.0,86.273533,STRONG BUY,82.301380,1.940045,-19.637675
70,RES,Resilient REIT Limited,Finance,8369.0,84.051398,STRONG BUY,40.513583,1.499274,-11.876968
157,YYLBEE,YeboYethu Limited,Finance,5400.0,82.150127,STRONG BUY,116.117092,1.787798,-25.454545
131,SEA,Spear REIT Ltd.,Finance,1300.0,81.957519,STRONG BUY,34.122397,0.897330,-13.037037
76,APH,Alphamin Resources Corp.,Non-energy minerals,1700.0,81.117503,STRONG BUY,50.587470,0.868800,-22.888889
43,PAN,Pan African Resources PLC,Non-energy minerals,3087.0,78.710603,STRONG BUY,181.462241,2.292087,-26.305131
74,HYP,Hyprop Investments Limited,Finance,5626.0,76.975255,STRONG BUY,30.252667,1.071749,-16.754655
169,ART,Argent Industrial Limited,Producer manufacturing,3760.0,76.800036,STRONG BUY,48.741618,1.367681,-8.160281
0,BHG,BHP Group Ltd,Non-energy minerals,70413.0,75.935488,STRONG BUY,55.173633,1.485114,-15.783455
69,FFB,Fortress Real Estate Investments Limited Class B,Finance,2586.0,74.478969,STRONG BUY,31.167268,1.034535,-16.822981


## 9. Value Trap Detection
Stocks that LOOK cheap (low P/E) but have negative momentum or poor Sharpe.


In [10]:
traps = liquid[
    (liquid["pe_ratio"].notna()) & (liquid["pe_ratio"] < 10) & (liquid["pe_ratio"] > 0) &
    ((liquid["mom_12m"] < -10) | (liquid["sharpe_1y"] < 0))
].copy()
trap_cols = ["symbol","description","sector","hist_price","pe_ratio","combined_score","signal","eps_growth_ttm","mom_12m","sharpe_1y"]
trap_avail = [c for c in trap_cols if c in traps.columns]
print(f"Value Traps: {len(traps)} -- AVOID these despite cheap P/E")
traps[trap_avail]


Value Traps: 30 -- AVOID these despite cheap P/E


,symbol,description,sector,hist_price,pe_ratio,combined_score,signal,eps_growth_ttm,mom_12m,sharpe_1y
89,LTE,Lighthouse Properties Plc,Finance,753.0,7.418073,69.627652,STRONG BUY,49.140995,-10.354023,-1.343644
229,MTU,Mantengu Limited,Finance,37.0,0.311448,66.336692,STRONG BUY,822.360248,-32.727273,-0.124656
36,KIO,Kumba Iron Ore Limited,Non-energy minerals,31541.0,6.899314,64.935241,BUY,-0.573361,-1.012579,-0.167592
135,HDC,Hudaco Industries Limited,Distribution services,19250.0,9.800534,64.827766,BUY,11.981606,-0.592251,-0.105165
8,NPN,Naspers Limited Class N,Technology services,86842.0,6.884510,63.815642,BUY,85.428004,-18.225315,-0.889770
5,PRX,Prosus N.V. Class N,Technology services,76300.0,7.092256,63.094669,BUY,91.409327,-18.152230,-1.009682
16,INPR,Investec Limited Non-Red.Non-Cum.Non-Ptg.Prf.Shs,Finance,9459.0,5.354294,62.406409,BUY,3.084839,-4.352856,-0.441412
199,SZK,SAB Zenzele Kabili Holdings (RF) Limited,Commercial services,3400.0,1.904079,58.955218,BUY,NaN,-11.678527,0.988944
62,SRE,Sirius Real Estate Limited,Finance,2190.0,7.935274,58.173651,BUY,39.855016,-0.288189,-0.328783
37,NRP,NEPI Rockcastle N.V,Finance,14176.0,9.992646,56.821064,BUY,-18.578181,1.806533,-0.257588


## 10. Oversold Opportunities (RSI < 30)


In [11]:
oversold = liquid[liquid["t_rsi_14"] < 30].sort_values("combined_score", ascending=False)
os_cols = ["symbol","description","sector","hist_price","t_rsi_14","combined_score","signal","mom_3m","sharpe_1y"]
os_avail = [c for c in os_cols if c in oversold.columns]
print(f"Oversold: {len(oversold)} stocks -- focus on those with high combined score")
oversold[os_avail]


Oversold: 20 stocks -- focus on those with high combined score


,symbol,description,sector,hist_price,t_rsi_14,combined_score,signal,mom_3m,sharpe_1y
134,RBO,Rainbow Chicken Limited,Process industries,600.0,27.044025,69.535108,STRONG BUY,-14.267804,1.388188
31,INP,Investec plc,Finance,13317.0,26.853377,63.330536,BUY,-2.360877,0.283266
17,INL,Investec Limited,Finance,13125.0,24.149433,63.109877,BUY,-3.165117,0.217689
66,AVI,AVI Limited Class Y,Consumer non-durables,9465.0,29.323308,59.383631,BUY,-15.553783,-0.289592
92,WBC,We Buy Cars Holdings Ltd.,Retail trade,3513.0,26.881720,54.702202,HOLD,-16.832386,-0.884123
54,SNT,Santam Limited,Finance,38300.0,29.208720,53.015165,HOLD,-15.159810,-0.829942
107,ITE,Italtile Limited,Retail trade,797.0,27.607362,51.009610,HOLD,-20.517045,-0.960769
192,BCF,Bowler Metcalf Limited,Producer manufacturing,1330.0,20.000000,50.865321,HOLD,-1.607390,0.991078
84,MTH,Motus Holdings Limited,Distribution services,10100.0,24.306473,50.407308,HOLD,-23.245086,0.183511
235,LAB,Labat Africa Ltd,Commercial services,3.0,0.000000,48.938992,HOLD,-62.500000,0.638390


## 11. Sector Analysis


In [12]:
sector_stats = liquid.groupby("sector").agg(
    count=("symbol","count"), avg_score=("combined_score","mean"),
    avg_sharpe=("sharpe_1y","mean"), avg_mom=("mom_12m","mean"),
    buys=("signal", lambda x: x.isin(["STRONG BUY","BUY"]).sum()),
).sort_values("avg_score", ascending=False)
sector_stats["buy_pct"] = (sector_stats["buys"]/sector_stats["count"]*100).round(0)
sector_stats


,count,avg_score,avg_sharpe,avg_mom,buys,buy_pct
sector,,,,,,
Consumer durables,2,66.969650,-0.207826,-1.036614,2,100.0
Finance,70,61.642985,0.488284,122.620831,50,71.0
Consumer services,10,60.012438,0.265953,6.293787,7,70.0
Non-energy minerals,29,58.183662,0.819205,63.010333,18,62.0
Communications,4,57.987717,0.832764,29.126112,3,75.0
Transportation,7,56.725249,0.128730,2.398126,4,57.0
Technology services,10,56.347229,-0.271048,-8.681438,6,60.0
Consumer non-durables,8,55.744431,0.194646,11.810558,4,50.0
Commercial services,11,54.130627,0.541976,1.634489,5,45.0


## 12. Deep Dive — Top 5 Picks


In [13]:
for i, (_, row) in enumerate(liquid.head(5).iterrows(), 1):
    sym = row["symbol"]
    name = row.get("description", "N/A")
    p = row.get("hist_price", row.get("price", 0))
    print(f"\n{'='*60}")
    print(f"#{i} {sym} | {name}")
    print(f"Price: R{p:,.0f} | Score: {row['combined_score']:.1f} | {row['signal']}")
    print(f"{'='*60}")
    print(f"  FUNDAMENTALS ({row['fundamental_score']:.0f}/100)")
    eg = row.get("eps_growth_ttm", 0)
    roe = row.get("roe_ttm", 0)
    nm = row.get("net_margin_ttm", 0)
    de = row.get("debt_to_equity", 0)
    print(f"    EPS Growth: {eg:.1f}%  |  ROE: {roe:.1f}%")
    print(f"    Net Margin: {nm:.1f}%  |  D/E: {de:.2f}")
    print(f"  TECHNICALS ({row['technical_score']:.0f}/100)")
    rsi = row.get("t_rsi_14", 0)
    sma50 = "Above" if row.get("above_sma50") else "Below"
    sma200 = "Above" if row.get("above_sma200") else "Below"
    mom = row.get("mom_12m", 0)
    macd_dir = "Bull" if row.get("t_macd_bullish") else "Bear"
    print(f"    RSI: {rsi:.0f}  |  SMA50: {sma50}  |  SMA200: {sma200}")
    print(f"    Mom 12m: {mom:.1f}%  |  MACD: {macd_dir}")
    print(f"  RISK ({row['risk_score']:.0f}/100)")
    sh = row.get("sharpe_1y", 0)
    dd = row.get("max_drawdown_1y", 0)
    vol = row.get("volatility_1y", 0)
    print(f"    Sharpe: {sh:.2f}  |  MaxDD: {dd:.1f}%  |  Vol: {vol:.1f}%")



#1 GND | Grindrod Limited
Price: R2,381 | Score: 86.3 | STRONG BUY
  FUNDAMENTALS (88/100)
    EPS Growth: 558.7%  |  ROE: 21.9%
    Net Margin: 38.5%  |  D/E: 0.33
  TECHNICALS (83/100)
    RSI: 83  |  SMA50: Above  |  SMA200: Above
    Mom 12m: 82.3%  |  MACD: Bull
  RISK (86/100)
    Sharpe: 1.94  |  MaxDD: -19.6%  |  Vol: 29.9%

#2 RES | Resilient REIT Limited
Price: R8,369 | Score: 84.1 | STRONG BUY
  FUNDAMENTALS (75/100)
    EPS Growth: 58.5%  |  ROE: 18.6%
    Net Margin: 116.4%  |  D/E: 0.52
  TECHNICALS (98/100)
    RSI: 57  |  SMA50: Above  |  SMA200: Above
    Mom 12m: 40.5%  |  MACD: Bull
  RISK (86/100)
    Sharpe: 1.50  |  MaxDD: -11.9%  |  Vol: 18.3%

#3 EQU | Equites Property Fund Ltd ZAR
Price: R1,814 | Score: 82.6 | STRONG BUY
  FUNDAMENTALS (80/100)
    EPS Growth: 65.2%  |  ROE: 11.3%
    Net Margin: 38.8%  |  D/E: 0.72
  TECHNICALS (92/100)
    RSI: 60  |  SMA50: Above  |  SMA200: Above
    Mom 12m: 18.9%  |  MACD: Bull
  RISK (74/100)
    Sharpe: 0.50  |  MaxDD:

## 13. Export Results


In [14]:
save_cols = ["symbol","sector","hist_price","fundamental_score","technical_score",
             "risk_score","combined_score","signal","mom_12m","sharpe_1y",
             "max_drawdown_1y","eps_growth_ttm","roe_ttm","net_margin_ttm",
             "revenue_growth_ttm","debt_to_equity","pe_ratio","avg_daily_value"]
save_cols = [c for c in save_cols if c in liquid.columns]
liquid[save_cols].to_csv(OUTPUT_DIR / f"jse_decision_{SNAPSHOT_DATE}.csv", index=False)
print(f"Saved: jse_decision_{SNAPSHOT_DATE}.csv ({len(liquid)} stocks)")
print(f"Blue Chips: {len(bluechip)} | Hidden Gems: {len(gems)} | Value Traps: {len(traps)}")


Saved: jse_decision_2026-05-15.csv (215 stocks)
Blue Chips: 33 | Hidden Gems: 51 | Value Traps: 30


## How to Use
1. Upload new TradingView snapshot to `data/snapshots/YYYY-MM-DD/`
2. Run `python download_bulk_historical.py` to refresh price data
3. Update `SNAPSHOT_DATE` in Cell 1
4. **Run All Cells**
5. Review Buy List → Blue Chips → Gems → Check Value Traps
